# Data Preprocessing pipeline

Goal:
1. Aggregate the data
2. Clean the dataset
3. Prepare to store at vector store databse (pinecone)


## 1. Aggregate Data

In [1]:
import os
import pandas as pd

folder_path = "dataset/01_restaurant_scraped/"

df_list = []

for file_name in os.listdir(folder_path):
    # Step 1: filter CSV files
    if file_name.endswith(".csv"):
        # Step 2: build full path
        full_path = os.path.join(folder_path, file_name)
        
        # Step 3: read file
        df = pd.read_csv(full_path)
        
        # Optional MRT name
        mrt_name = file_name.replace(".csv", "")
        df["mrt_name"] = mrt_name
        
        df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)

In [2]:
combined_df

,name,category,rating,reviews,price,address,status,snippet,img,url,postal,lon,lat,mrt_name
0,Yakiniku Like (The Clementi Mall),Yakiniku,4.7,"(1,666)",$10–20,"3155 Commonwealth Ave W, #05-35/36/37",Closed · Opens 11 am Sun,"""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,https://www.google.com/maps/place/Yakiniku+Lik...,129588.0,103.764725,1.314987,clementi
1,家羊地 - Jiayangdi - Bbq/烧烤 (Newest),Barbecue,4.7,(464),$20–30,"1 W Coast Dr, #01-97 NEWest",Open · Closes 2 am,"""Great",https://lh3.googleusercontent.com/gps-cs-s/AHV...,https://www.google.com/maps/place/%E5%AE%B6%E7...,169655.0,103.827405,1.289417,clementi
2,Captain Kim Korean BBQ & Hotpot Buffet,Buffet,4.8,"(5,921)",$20–30,"3151 Commonwealth Ave W, #02-05 Grantral Mall ...",Closed · Opens 11:30 am Sun,"""Best experience ever , the",https://lh3.googleusercontent.com/gps-cs-s/AHV...,https://www.google.com/maps/place/Captain+Kim+...,529653.0,103.941943,1.352336,clementi
3,Ayer Rajah Food Centre,Hawker center,4.2,"(7,328)",$1–10,503 W Coast Dr,Open · Closes 1 am,"""The braised duck is delicious.""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,https://www.google.com/maps/place/Ayer+Rajah+F...,120503.0,103.759751,1.311803,clementi
4,MEET Noodles 面面 Lanzhou Beef Noodle 兰州牛肉面,Self service restaurant,4.7,(928),$1–10,"3151 Commonwealth Ave W, #01-15 Grantral Mall ...",Closed · Opens 10:30 am Sun,"""Great",https://lh3.googleusercontent.com/gps-cs-s/AHV...,https://www.google.com/maps/place/MEET+Noodles...,129581.0,103.765147,1.314270,clementi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12634,思carrot cake,Hawker Stall,NaN,NaN,NaN,"85 Redhill Ln, #01-56",Hawker Stall,NaN,https://streetviewpixels-pa.googleapis.com/v1/...,https://www.google.com/maps/place/%E6%80%9Dcar...,599213,103.775215,1.340788,redhill
12635,Western Delights,Hawker Stall,3.9,(8),$1–10,"85 Redhill Ln, #01-83",Closed · Opens 11:30 am Sun,"""The fish and chips so crispy yet the fish ins...",https://lh3.googleusercontent.com/gps-cs-s/AHV...,https://www.google.com/maps/place/Western+Deli...,270020,103.788230,1.310997,redhill
12636,Ya Kun Kaya Toast (Artra),Cafe,3.5,(136),$1–10,"12 Alexandra View, #01-08",Closed · Opens 7:30 am Sun,"""Happy greeting to tasty",https://lh3.googleusercontent.com/gps-cs-s/AHV...,https://www.google.com/maps/place/Ya+Kun+Kaya+...,158745,103.817185,1.290031,redhill
12637,Redhill Katong Big Prawn Mee 红山加东大蝦麵,Hawker Stall,3.8,(33),$1–10,116 Bukit Merah Vw,Open 24 hours,"""He expertly cooked the prawn noodle soup and ...",https://lh3.googleusercontent.com/gps-cs-s/AHV...,https://www.google.com/maps/place/Redhill+Kato...,151116,103.821532,1.285134,redhill


## Save the data to aggregated folder

In [3]:
# Define the folder path and file name
folder_path = r'dataset/02_aggregated/'
file_name = 'aggregated_restaurant.csv'

# Create the directory if it doesn't exist
os.makedirs(folder_path, exist_ok=True)

# Join the path and filename
final_path = os.path.join(folder_path, file_name)

# Save the DataFrame
combined_df.to_csv(final_path, index=False, encoding='utf-8')

## Data cleaning pipeline

Process:
- Remove entries if it has NaN  value in name, category, lat, or lon.
- Remove duplicate entries if it has the same url.
- Remoce duplicate name-address pairs


### Remove NaN values

In [4]:
initial_len = len(combined_df)
print(f"Initial rows: {initial_len}")

combined_df = combined_df.dropna(subset=["name", "category", "lat", "lon"])
print(f"After Nan Removal: {len(combined_df)}")

print(f'\nRemoved NaN entries: {initial_len - len(combined_df)} rows')

Initial rows: 12639
After Nan Removal: 12235

Removed NaN entries: 404 rows


### Remove duplicate URLs

In [5]:
initial_len = len(combined_df)
print(f"Initial rows: {initial_len}")

combined_df = (
    combined_df
    .groupby("url", as_index=False)
    .agg({"name": "first", "category": "first", "rating": "first", "reviews": "first", "price": "first", "address": "first", "status": "first", "snippet": "first", "img": "first", "postal": "first", "lon": "first", "lat": "first",
          "mrt_name": lambda x: list(x.dropna().unique())})
)

# Rename mrt column
combined_df = combined_df.rename(columns={"mrt_name": "nearest_mrt"})

print(f"After deduplication: {len(combined_df)}")
print(f"Removed rows: {initial_len - len(combined_df)}")

Initial rows: 12235
After deduplication: 8376
Removed rows: 3859


### Remove duplicate name-address pairs

In [6]:
initial_len = len(combined_df)
print(f"Initial rows: {initial_len}")

combined_df["name_norm"] = combined_df["name"].str.upper().str.strip()
combined_df["address_norm"] = combined_df["address"].str.upper().str.strip()

duplicates = combined_df[
    combined_df.duplicated(subset=["name_norm", "address_norm"], keep=False)
]

print("\n=== Duplicate (name, address) rows ===")
print(duplicates)

# drop duplicates
combined_df = combined_df.drop_duplicates(
    subset=["name_norm", "address_norm"],
    keep="first"
)

# Step 3: report results
final_len = len(combined_df)

print(f"\nAfter duplicate drop: {final_len}")
print(f"Removed duplicate rows: {initial_len - final_len}")

Initial rows: 8376

=== Duplicate (name, address) rows ===
                                                    url  \
8249  https://www.google.com/maps/place/Zaffron+Tand...   
8250  https://www.google.com/maps/place/Zaffron+Tand...   

                                                   name    category  rating  \
8249  Zaffron Tandoori - North Indian and Mexican Re...  Restaurant     4.8   
8250  Zaffron Tandoori - North Indian and Mexican Re...  Restaurant     4.8   

     reviews price       address                              status  \
8249   (240)  None  46 Boat Quay  Open · Closes 2 am · Reopens 11 am   
8250   (240)  None  46 Boat Quay  Open · Closes 2 am · Reopens 11 am   

            snippet                                                img  \
8249  "Outstanding   https://lh3.googleusercontent.com/gps-cs-s/AHV...   
8250        Dine-in  https://lh3.googleusercontent.com/p/AF1QipPS1T...   

       postal         lon       lat                       nearest_mrt  \
8249  49835.

In [7]:
combined_df.head()

,url,name,category,rating,reviews,price,address,status,snippet,img,postal,lon,lat,nearest_mrt,name_norm,address_norm
0,https://www.google.com/maps/place/$1.50+Dim+Su...,$1.50 Dim Sum Bedok,Hawker Stall,3.7,(479),$1.50,"Block 123 Bedok North Street 2, 01-142",Open 24 hours,"""Very reasonable price n very delicious""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,460123.0,103.937343,1.329199,"[bedok_north, bedok_reservoir]",$1.50 DIM SUM BEDOK,"BLOCK 123 BEDOK NORTH STREET 2, 01-142"
1,https://www.google.com/maps/place/%2301-12+Han...,#01-12 Handmade Noodles,Hawker Stall,4.4,(12),$1–10,"4A Eunos Cres, #01-99 4A",Hawker Stall,"""It tastes not bad at all, extra crispy one si...",https://lh3.googleusercontent.com/gps-cs-s/AHV...,402004,103.904256,1.320331,[eunos],#01-12 HANDMADE NOODLES,"4A EUNOS CRES, #01-99 4A"
2,https://www.google.com/maps/place/%2301-18+%E5...,#01-18 廣東砂煲飯。小吃,Hawker Stall,4.0,(5),$1–10,"#01-18 Geylang Bahru, Market & Food Centre",Hawker Stall,"""Very authentic and good claypot rice.""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,330069.0,103.870005,1.321463,[geylang_bahru],#01-18 廣東砂煲飯。小吃,"#01-18 GEYLANG BAHRU, MARKET & FOOD CENTRE"
3,https://www.google.com/maps/place/%2301-20+%E8...,#01-20 萝卜糕 经济小吃,Hawker Stall,4.7,(11),$1–10,"22A Havelock Rd, #01-20",Closed · Opens 8 am,"""Fried till crispy yet not greasy.""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,161022.0,103.829623,1.287971,[havelock],#01-20 萝卜糕 经济小吃,"22A HAVELOCK RD, #01-20"
4,https://www.google.com/maps/place/%2301-35+%E5...,#01-35 健康齋素食,Hawker Stall,5.0,(1),None,"79 Telok Blangah Dr, #01-35, Telok Blangah Foo...",Closed · Opens 6 am,Takeaway,NaN,100079.0,103.807618,1.273356,[telok_blangah],#01-35 健康齋素食,"79 TELOK BLANGAH DR, #01-35, TELOK BLANGAH FOO..."


### Remove or Map noisy categories

In [8]:
combined_df['category'].value_counts()


category
Hawker Stall                              1516
Restaurant                                1483
Chinese                                    593
Japanese                                   305
Indian                                     238
                                          ... 
Soft drinks shop                             1
Takoyaki restaurant                          1
Educational institution                      1
Sweets and dessert buffet                    1
Unfussy rooms, plus a restaurant & bar       1
Name: count, Length: 295, dtype: int64

In [9]:
combined_df[combined_df['category'] == 'Educational institution']

,url,name,category,rating,reviews,price,address,status,snippet,img,postal,lon,lat,nearest_mrt,name_norm,address_norm
8242,https://www.google.com/maps/place/Yusof+Ishak+...,Yusof Ishak House,Educational institution,4.1,(64),None,31 Lower Kent Ridge Rd,Closed · Opens 9 am,"""They were excellent and delicious.""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,119078.0,103.7748,1.298572,[kent_ridge],YUSOF ISHAK HOUSE,31 LOWER KENT RIDGE RD


In [10]:
import json

# normalize text
combined_df["category_clean"] = (combined_df["category"].astype(str).str.lower().str.strip())

with open("category_map.json", "r") as f:
    exact_map = json.load(f)

# Keyword rules
# keyword_rules = [
#     (["japanese", "sushi", "ramen", "takoyaki"], "japanese"),
#     (["chinese"], "chinese"),
#     (["indian"], "indian"),
#     (["hawker"], "local"),
#     (["cafe", "coffee"], "cafe"),
#     (["dessert", "sweet"], "dessert"),
# ]

# mapping function
def map_category(cat):
    cat = str(cat).lower()

    # check exact match
    if cat in exact_map:
        return exact_map[cat]

    # check keyword rules
    # for keywords, label in keyword_rules:
    #     for keyword in keywords:
    #         if keyword in cat:
    #             return label

    # fallback
    return cat

# apply mapping
combined_df["category_group"] = combined_df["category_clean"].apply(map_category)

# check results
pd.set_option("display.max_rows", None)
print(combined_df["category_group"].value_counts())

category_group
hawker                                    1661
restaurant                                1520
chinese                                    728
japanese                                   520
cafe                                       405
indian                                     315
fast_food                                  265
italian                                    200
thai                                       196
korean                                     193
halal                                      174
seafood                                    169
western                                    152
vegetarian                                 142
asian                                      107
food_court                                  77
bistro                                      72
mall                                        72
singaporean                                 70
taiwanese                                   65
vietnamese                                  6

In [11]:
pd.reset_option("display.max_rows")
combined_df

,url,name,category,rating,reviews,price,address,status,snippet,img,postal,lon,lat,nearest_mrt,name_norm,address_norm,category_clean,category_group
0,https://www.google.com/maps/place/$1.50+Dim+Su...,$1.50 Dim Sum Bedok,Hawker Stall,3.7,(479),$1.50,"Block 123 Bedok North Street 2, 01-142",Open 24 hours,"""Very reasonable price n very delicious""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,460123.0,103.937343,1.329199,"[bedok_north, bedok_reservoir]",$1.50 DIM SUM BEDOK,"BLOCK 123 BEDOK NORTH STREET 2, 01-142",hawker stall,hawker
1,https://www.google.com/maps/place/%2301-12+Han...,#01-12 Handmade Noodles,Hawker Stall,4.4,(12),$1–10,"4A Eunos Cres, #01-99 4A",Hawker Stall,"""It tastes not bad at all, extra crispy one si...",https://lh3.googleusercontent.com/gps-cs-s/AHV...,402004,103.904256,1.320331,[eunos],#01-12 HANDMADE NOODLES,"4A EUNOS CRES, #01-99 4A",hawker stall,hawker
2,https://www.google.com/maps/place/%2301-18+%E5...,#01-18 廣東砂煲飯。小吃,Hawker Stall,4.0,(5),$1–10,"#01-18 Geylang Bahru, Market & Food Centre",Hawker Stall,"""Very authentic and good claypot rice.""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,330069.0,103.870005,1.321463,[geylang_bahru],#01-18 廣東砂煲飯。小吃,"#01-18 GEYLANG BAHRU, MARKET & FOOD CENTRE",hawker stall,hawker
3,https://www.google.com/maps/place/%2301-20+%E8...,#01-20 萝卜糕 经济小吃,Hawker Stall,4.7,(11),$1–10,"22A Havelock Rd, #01-20",Closed · Opens 8 am,"""Fried till crispy yet not greasy.""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,161022.0,103.829623,1.287971,[havelock],#01-20 萝卜糕 经济小吃,"22A HAVELOCK RD, #01-20",hawker stall,hawker
4,https://www.google.com/maps/place/%2301-35+%E5...,#01-35 健康齋素食,Hawker Stall,5.0,(1),None,"79 Telok Blangah Dr, #01-35, Telok Blangah Foo...",Closed · Opens 6 am,Takeaway,NaN,100079.0,103.807618,1.273356,[telok_blangah],#01-35 健康齋素食,"79 TELOK BLANGAH DR, #01-35, TELOK BLANGAH FOO...",hawker stall,hawker
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8371,https://www.google.com/maps/place/umisushi+Mou...,umisushi Mount Alvernia Hospital,Japanese,3.6,(59),$10–20,"820 Thomson Road, #01-27, Mount Alvernia Hospital",Closed · Opens 10 am Mon,"""Her warmth and attentiveness truly made my",https://lh3.googleusercontent.com/gps-cs-s/AHV...,574623,103.837774,1.341499,[caldecott],UMISUSHI MOUNT ALVERNIA HOSPITAL,"820 THOMSON ROAD, #01-27, MOUNT ALVERNIA HOSPITAL",japanese,japanese
8372,https://www.google.com/maps/place/umisushi+One...,umisushi One Raffles Place,Sushi takeaway,3.1,(87),$10–20,"1 Raffles Pl, #B1 - 24 / 25",Closed · Opens 10 am Mon,"""Good service, quality",https://lh3.googleusercontent.com/gps-cs-s/AHV...,48616,103.851073,1.284350,[raffles_place],UMISUSHI ONE RAFFLES PLACE,"1 RAFFLES PL, #B1 - 24 / 25",sushi takeaway,japanese
8373,https://www.google.com/maps/place/umisushi/dat...,umisushi,Japanese,3.7,(42),$1–10,"3 Simei Street 6, B1-K7 Eastpoint Mall",Closed · Opens 10 am Sun,"""Top-notch service, kind and friendly staff, a...",https://lh3.googleusercontent.com/gps-cs-s/AHV...,278995.0,103.796399,1.312240,[simei],UMISUSHI,"3 SIMEI STREET 6, B1-K7 EASTPOINT MALL",japanese,japanese
8374,https://www.google.com/maps/place/waters+edge+...,waters edge Training Restaurant,Restaurant,3.2,(30),None,15 Woodlands Ave 9,Open 24 hours,"""It’s a training school for students joining t...",https://lh3.googleusercontent.com/gps-cs-s/AHV...,738967.0,103.783018,1.445412,"[admiralty, woodlands]",WATERS EDGE TRAINING RESTAURANT,15 WOODLANDS AVE 9,restaurant,restaurant


In [12]:
# Define the folder path and file name
folder_path = r'dataset/03_cleaned/'
file_name = 'cleaned_restaurant.csv'

# Create the directory if it doesn't exist
os.makedirs(folder_path, exist_ok=True)

# Join the path and filename
final_path = os.path.join(folder_path, file_name)

# Save the DataFrame
combined_df.to_csv(final_path, index=False, encoding='utf-8')